In [ ]:
from pathlib import Path

beyer2020_air = str(Path.cwd() / "data" / "Beyer2020" / "data" / "LateQuaternary_Environment.nc")
palmod2_air = str(Path.cwd() / "data" / "PalMod2" / "output" / "PMMXMCRTDGr111Amtasgn30201_1-250.nc")
palmod2_soil = str(Path.cwd() / "data" / "PalMod2" / "output" / "PMMXMCRTDGr111Lmtslgn30201_1-250.nc")
trace_air = str(Path.cwd() / "data" / "TraCE-21k" / "data" / "trace.01-36.22000BP.clm2.TSA.22000BP_decavg_400BCE.nc")
trace_soil = str(Path.cwd() / "data" / "TraCE-21k" / "data" / "trace.01-36.22000BP.clm2.TSOI.22000BP_decavg_400BCE.nc")
chelsa_air = str(Path.cwd() / "data" / "CHELSA-TraCE21k-centennial" / "output" / "tasmean.nc")


In [ ]:
import pandas as pd

df_air = pd.DataFrame(columns=["name", "path", "variable", "unit"])
df_air.loc[0] = ["Beyer2020", beyer2020_air, "temperature", "degC"]
df_air.loc[1] = ["PalMod2", palmod2_air, "tas", "K"]
df_air.loc[2] = ["TraCE-21k", trace_air, "TSA", "K"]
df_air.loc[3] = ["CHELSA-TraCE21k-centennial", chelsa_air, "tasmean", "K"]

df_air

In [ ]:
df_soil = pd.DataFrame(columns=["name", "path", "variable", "unit"])
df_soil.loc[0] = ["PalMod2", palmod2_soil, "tsl", "K"]
df_soil.loc[1] = ["TraCE-21k", trace_soil, "TSOI", "K"]

df_soil

In [ ]:
import pandas as pd
import numpy as np
import xarray as xr
from pathlib import Path


def inspect_dataset(name, path, variable, unit_in_df):
    """Return a multi-line string describing one NetCDF dataset."""
    lines = []
    lines.append("=" * 78)
    lines.append(f"DATASET: {name}")
    lines.append("=" * 78)
    lines.append(f"path     : {path}")
    lines.append(f"variable : {variable}   (declared unit in df_air: {unit_in_df})")
    lines.append("")

    with xr.open_dataset(path, decode_times=False) as ds:
        # --- global attrs --------------------------------------------------
        lines.append("-- global attributes --")
        if ds.attrs:
            for k, v in ds.attrs.items():
                lines.append(f"  {k}: {v}")
        else:
            lines.append("  (none)")
        lines.append("")

        # --- dimensions ----------------------------------------------------
        lines.append("-- dimensions --")
        for d, n in ds.dims.items():
            lines.append(f"  {d}: {n}")
        lines.append("")

        # --- coordinates ---------------------------------------------------
        lines.append("-- coordinates --")
        for cname in ds.coords:
            c = ds[cname]
            vals = np.asarray(c.values)
            try:
                rng = f"[{float(vals.min()):.4f}, {float(vals.max()):.4f}]"
            except (TypeError, ValueError):
                rng = f"first={vals.flat[0]}, last={vals.flat[-1]}"
            lines.append(f"  {cname}  shape={c.shape}  dtype={c.dtype}  range={rng}")
            for ak, av in c.attrs.items():
                lines.append(f"      {ak}: {av}")
        lines.append("")

        # --- time convention (the part that actually broke the plot) -------
        if "time" in ds.coords:
            t = ds["time"]
            tvals = np.asarray(t.values, dtype=float)
            tunits = t.attrs.get("units", "<missing>")
            # infer convention
            tu_low = tunits.lower()
            if "since" in tu_low:
                conv = "CF style ('... since ...') — past is NEGATIVE, ka BP = -time/1000"
            elif "ka" in tu_low or "kyr" in tu_low:
                conv = "kiloyears BP — past is POSITIVE, already in ka BP"
            elif "yr" in tu_low or "year" in tu_low:
                conv = "years BP — past is POSITIVE, ka BP = time/1000"
            else:
                conv = ("UNKNOWN units string — fallback: "
                        + ("treat as CF (neg→past)" if tvals.min() < 0 else "treat as yr BP"))
            step = np.diff(tvals)
            step_info = (f"median step = {np.median(step):.3f}, "
                         f"min = {step.min():.3f}, max = {step.max():.3f}"
                         if step.size else "n/a")
            lines.append("-- time convention --")
            lines.append(f"  units    : {tunits}")
            lines.append(f"  range    : [{tvals.min():.3f}, {tvals.max():.3f}]  (n={tvals.size})")
            lines.append(f"  step     : {step_info}")
            lines.append(f"  interpretation: {conv}")
            lines.append("")

        # --- the temperature variable -------------------------------------
        lines.append(f"-- variable: {variable} --")
        if variable in ds.variables:
            v = ds[variable]
            lines.append(f"  dims    : {v.dims}")
            lines.append(f"  shape   : {v.shape}")
            lines.append(f"  dtype   : {v.dtype}")
            for ak, av in v.attrs.items():
                lines.append(f"  attr {ak}: {av}")
        else:
            lines.append(f"  !! variable {variable!r} NOT FOUND in file.")
            lines.append(f"  available data vars: {list(ds.data_vars)}")
        lines.append("")

        # --- lat / lon detection (what plot_temperature_comparison uses) ---
        LAT_NAMES = ["lat", "latitude", "nav_lat", "y"]
        LON_NAMES = ["lon", "longitude", "nav_lon", "x"]
        lat_name = next((n for n in LAT_NAMES if n in ds.coords), None)
        lon_name = next((n for n in LON_NAMES if n in ds.coords), None)
        lines.append("-- spatial grid --")
        lines.append(f"  detected lat coord: {lat_name}")
        lines.append(f"  detected lon coord: {lon_name}")
        if lat_name and lon_name:
            la = ds[lat_name].values
            lo = ds[lon_name].values
            lon_conv = "0–360" if float(lo.max()) > 180 else "-180–180"
            lines.append(f"  lat range : [{float(la.min()):.3f}, {float(la.max()):.3f}]"
                         f"  resolution ≈ {float(np.median(np.abs(np.diff(la)))):.4f}°")
            lines.append(f"  lon range : [{float(lo.min()):.3f}, {float(lo.max()):.3f}]"
                         f"  resolution ≈ {float(np.median(np.abs(np.diff(lo)))):.4f}°  "
                         f"(convention: {lon_conv})")
        lines.append("")

    return "\n".join(lines)


# --- run inspection across df_air, print + save ----------------------------
out_path = Path("dataset_info.txt")
report = []

for _, row in df_air.iterrows():
    block = inspect_dataset(row["name"], row["path"], row["variable"], row["unit"])
    report.append(block)
    #print(block)

out_path.write_text("\n".join(report), encoding="utf-8")
#print(f"\n→ Saved full report to: {out_path.resolve()}")

In [ ]:
import pandas as pd
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt


def time_to_ka_bp(time_da):
    """Convert NetCDF time coord to ka BP (past = positive, present = 0)."""
    units = (time_da.attrs.get("units") or "").lower()
    vals = np.asarray(time_da.values, dtype=float)

    # "ka BP" / "kyr BP" — explicit kiloyears (may be signed: TraCE uses negatives)
    if "ka" in units or "kyr" in units:
        return -vals if vals.min() < 0 else vals

    # "years since present" — past is negative (Beyer2020)
    if "since present" in units:
        return -vals / 1000

    # "days since <date>" — simulation time, assume last sample = present
    if "since" in units and "day" in units:
        return (vals.max() - vals) / 365250.0

    # "years since <date>" — same trick in years
    if "since" in units:
        return (vals.max() - vals) / 1000.0

    # plain "yr BP" — positive values
    if "yr" in units or "year" in units:
        return vals / 1000

    # last‑resort fallback: sign tells you the convention
    return (-vals / 1000) if vals.min() < 0 else (vals / 1000)

def extract_temperature_at_point(PATH, temp_var, lat, lon, input_unit="degC"):
    """Return (time_ka, y_degC, cell_lat, cell_lon) for one NetCDF file."""
    if input_unit not in ("degC", "K"):
        raise ValueError(f"input_unit must be 'degC' or 'K', got {input_unit!r}")

    LAT_NAMES = ["lat", "latitude", "nav_lat", "y"]
    LON_NAMES = ["lon", "longitude", "nav_lon", "x"]

    with xr.open_dataset(PATH, decode_times=False) as ds:
        lat_name = next((n for n in LAT_NAMES if n in ds.coords), None)
        lon_name = next((n for n in LON_NAMES if n in ds.coords), None)
        if lat_name is None or lon_name is None:
            raise KeyError(f"Could not find lat/lon coords in {list(ds.coords)}")

        # handle 0–360 vs -180–180 longitude conventions
        lon_q = lon + 360 if float(ds[lon_name].max()) > 180 and lon < 0 else lon

        da = ds[temp_var].sel({lat_name: lat, lon_name: lon_q}, method="nearest")
        cell_lat = float(da[lat_name])
        cell_lon = float(da[lon_name])

        if "month" in da.dims:           # collapse seasonal cycle → annual mean
            da = da.mean(dim="month")
        if input_unit == "K":            # K → °C is just an offset
            da = da - 273.15

        time_ka = time_to_ka_bp(ds["time"])
        y = da.values.copy()

    return time_ka, y, cell_lat, cell_lon


def plot_temperature_comparison(df, lat, lon, title=None, log_x=True):
    """Overlay temperature time series from several NetCDF files on one axis."""
    fig, ax = plt.subplots(figsize=(11, 5))
    results = {}

    for _, row in df.iterrows():
        time_ka, y, cell_lat, cell_lon = extract_temperature_at_point(
            row["path"], row["variable"], lat, lon, input_unit=row["unit"]
        )
        ax.plot(time_ka, y, lw=1.3,
                label=f"{row['name']}  ({cell_lat:.2f}°N, {cell_lon:.2f}°E)")
        results[row["name"]] = (time_ka, y)

    if log_x:
        # symlog: linear within ±0.1 ka (last 100 yr), log beyond
        ax.set_xscale("symlog", linthresh=0.1)
        ax.set_xticks([0, 0.1, 1, 10, 100])
        ax.set_xticklabels(["0", "0.1", "1", "10", "100"])

    ax.invert_xaxis()                                  # present on the right, past on the left
    ax.axhline(0, color="k", lw=0.5, ls="--", alpha=0.5)
    ax.set_xlabel("Time (ka BP)")
    ax.set_ylabel("Temperature (°C)")
    ax.set_title(title or f"Air temperature at ({lat:.2f}°N, {lon:.2f}°E)")
    ax.grid(alpha=0.3, which="both")
    ax.legend()
    plt.tight_layout()
    plt.show()
    return results


# --- quick diagnostic: print each file's time convention before plotting -----
for _, row in df_air.iterrows():
    with xr.open_dataset(row["path"], decode_times=False) as ds:
        t = ds["time"]
        print(f"{row['name']:30s} units={t.attrs.get('units')!r:45s} "
              f"range=[{float(t.min()):.1f}, {float(t.max()):.1f}]  n={t.size}")

# --- run --------------------------------------------------------------------
results = plot_temperature_comparison(df_air, lat=52.52, lon=13.40)  # Berlin

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle


def collect_summary(df, lat_req, lon_req):
    """Pull just the fields we need for the visual summary."""
    rows = []
    LAT_NAMES = ["lat", "latitude", "nav_lat", "y"]
    LON_NAMES = ["lon", "longitude", "nav_lon", "x"]

    for _, r in df.iterrows():
        with xr.open_dataset(r["path"], decode_times=False) as ds:
            ka = np.sort(time_to_ka_bp(ds["time"]))    # ← same helper as the plot cell
            step_kyr = float(np.median(np.diff(ka))) if ka.size > 1 else np.nan

            # grid cell that .sel(method="nearest") would pick
            lat_name = next(n for n in curl -fsSL https://claude.ai/install.sh | bashLAT_NAMES if n in ds.coords)
            lon_name = next(n for n in LON_NAMES if n in ds.coords)
            lon_q = lon_req + 360 if float(ds[lon_name].max()) > 180 and lon_req < 0 else lon_req
            sub = ds[r["variable"]].sel({lat_name: lat_req, lon_name: lon_q}, method="nearest")
            cell_lat = float(sub[lat_name])
            cell_lon = float(sub[lon_name])
            if cell_lon > 180:                         # display in -180..180
                cell_lon -= 360
            lat_res = float(np.median(np.abs(np.diff(ds[lat_name].values))))
            lon_res = float(np.median(np.abs(np.diff(ds[lon_name].values))))

        rows.append({
            "name": r["name"],
            "t_min": float(ka.min()), "t_max": float(ka.max()),
            "n_steps": int(ka.size), "step_kyr": step_kyr,
            "cell_lat": cell_lat, "cell_lon": cell_lon,
            "lat_res": lat_res, "lon_res": lon_res,
        })
    return rows


def plot_dataset_summary(df, lat_req, lon_req):
    rows = collect_summary(df, lat_req, lon_req)
    colors = plt.get_cmap("tab10").colors
    fig, (ax_t, ax_s) = plt.subplots(
        1, 2, figsize=(13, 0.55 * len(rows) + 2),
        gridspec_kw={"width_ratios": [2.2, 1]},
    )

    # ---- panel 1: temporal coverage on symlog axis -----------------------
    for i, r in enumerate(rows):
        ax_t.hlines(i, r["t_min"], r["t_max"], colors=colors[i], lw=6)
        ax_t.text(r["t_max"], i + 0.25,
                  f"  Δt ≈ {r['step_kyr']*1000:.0f} yr   n={r['n_steps']}",
                  va="bottom", ha="left", fontsize=9, color="gray")
    ax_t.set_yticks(range(len(rows)))
    ax_t.set_yticklabels([r["name"] for r in rows])
    ax_t.set_xscale("symlog", linthresh=0.1)
    ax_t.set_xticks([0, 0.1, 1, 10, 100])
    ax_t.set_xticklabels(["0", "0.1", "1", "10", "100"])
    ax_t.invert_xaxis()
    ax_t.invert_yaxis()
    ax_t.set_xlabel("Time (ka BP)")
    ax_t.set_title("Temporal coverage")
    ax_t.grid(alpha=0.3, which="both", axis="x")

    # ---- panel 2: requested point vs. selected grid cell -----------------
    ax_s.scatter([lon_req], [lat_req], marker="*", s=200,
                 color="k", zorder=5, label="requested")
    for i, r in enumerate(rows):
        ax_s.add_patch(Rectangle(
            (r["cell_lon"] - r["lon_res"] / 2, r["cell_lat"] - r["lat_res"] / 2),
            r["lon_res"], r["lat_res"],
            edgecolor=colors[i], facecolor=colors[i], alpha=0.25, lw=1.2,
            label=f"{r['name']}  ({r['lat_res']:.2f}°×{r['lon_res']:.2f}°)",
        ))
        ax_s.scatter([r["cell_lon"]], [r["cell_lat"]],
                     color=colors[i], s=30, zorder=4)
    ax_s.set_xlabel("Longitude (°E)")
    ax_s.set_ylabel("Latitude (°N)")
    ax_s.set_title(f"Grid cell selected near ({lat_req:.2f}°N, {lon_req:.2f}°E)")
    ax_s.set_aspect("equal", adjustable="datalim")
    ax_s.grid(alpha=0.3)
    ax_s.legend(fontsize=8, loc="best")

    plt.tight_layout()
    plt.show()
    return rows


summary = plot_dataset_summary(df_air, lat_req=52.52, lon_req=13.40)  # Berlin

In [ ]:
# cdo approach cannot run even after optimization because 32gb of ram is not enough

import subprocess
from pathlib import Path
output_dir = Path.cwd() / 'data' / 'CHELSA-TraCE21k-centennial' / 'output'
tasmean_nc = output_dir / 'tasmean.nc'
tasmean_nc.unlink(missing_ok=True)

print("Computing tasmean from tasmax and tasmin...")
cmd = [
    'cdo',
    '-P', '1',              # single thread = lowest memory
    '-z', 'zip_5',
    '-b', 'F32',
    '-setname,tasmean',
    '-mulc,0.5',
    '-add',
    'tasmin.nc',
    'tasmax.nc',
    'tasmean.nc',
]
result = subprocess.run(cmd, capture_output=True, text=True, cwd=str(output_dir))
print(result.stderr)
print(f'Exit code: {result.returncode}')
print(f'Output file: {tasmean_nc} (exists: {tasmean_nc.exists()})')